# Gain-prior + waveshaper on **LA2A** — Local Eval

Companion to [`train_lstm_gain_prior_ws.ipynb`](train_lstm_gain_prior_ws.ipynb)
(this folder) — the LA2A retarget of the diffssl waveshaper gain-prior. Same
`GainPriorWSDiffSSLLSTM` model and the SAME 9-column streaming metric engine as
`06_output/eval_lstm_gain_prior_ws.ipynb`, so LA2A numbers drop straight into the
cross-model comparison. Two things differ from the diffssl eval:

- **GR on-the-fly.** Instead of loading `gr_curves/*.pt`, GR is recomputed per
  segment from `(dry, wet)` with the exact export function
  `src.dsp_torch.gain_reduction_db(·, 1024)` + a 1023-sample lookback — bit-identical
  to the exported curve (see `dataset_la2a`). No `.pt` files needed.
- **Split = temporal regions, not held-out pairs.** LA2A's 84 recordings each hold
  one setting, so the train notebook splits **within each recording** (`train[0,.8)
  / val[.8,.9) / test[.9,1)`). "Validation" / "test" here therefore mean the
  `[.8,.9)` / `[.9,1)` fraction **regions of every recording** — unseen audio at
  known settings. Each region is streamed statefully (reset once at region start,
  one warm-up chunk, LSTM state carried across chunks).

```
raw dry x ────────────────────────────────┐
x·g  (amplitude-matched, g = 10^(gr/20)) ─┤
gr   (reduction-positive, ~[0,1]) ────────┼─ cat → main LSTM(19→32)
tvcond cond_seq[16] (pool(|x|)⊕knobs[2]) ─┘          │
                                   ┌─────────────────┴──────────────┐
                          Δg = 12·tanh(lin_g(h))          c = lin_c(h)
                     s = x · 10^((gr + Δg)/20)                 │
                     y = W(s) + c,   W(s) = s + r(s) − r(0)    │
```

- **Static knobs**: `[comp_limit, peak_reduction]` via `TVFiLMCond` (SOTA `tvcond`)
- **Target**: wet audio; **Stateful inference**: whole-region streaming
- **GR-source ablation** (§9): oracle / const-mean / const-0 dB / amp-match. No
  `predicted` variant — there is no LA2A GR predictor (05_conditioning's is diffssl-only).

In [ ]:
# ── 0. Imports and defaults ─────────────────────────────────────────

import gc, glob, json, os, sys, types, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

warnings.filterwarnings("ignore", category=UserWarning)

# model_gainprior_ws imports nablafx's TVFiLMCond (via model_tfilm). Stub the
# `rational` / `frechet_audio_distance` import-chain deps so `from nablafx...`
# doesn't drag in broken wheels — identical to the train notebook's cell 0.
_rational = types.ModuleType("rational")
_rational.torch = types.ModuleType("rational.torch")
_rational.torch.Rational = type("Rational", (), {})
sys.modules.setdefault("rational", _rational)
sys.modules.setdefault("rational.torch", _rational.torch)
_fad = types.ModuleType("frechet_audio_distance")
_fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
sys.modules.setdefault("frechet_audio_distance", _fad)

REPO_ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file()),
    Path.cwd().resolve(),
)
sys.path.insert(0, str(REPO_ROOT))                        # `src`
sys.path.insert(0, str(REPO_ROOT / "nablafx"))            # raw ./nablafx/ clone (TVFiLMCond)
sys.path.insert(0, str(REPO_ROOT / "06_output"))          # model_gainprior_ws, system_gainprior, amplitude_match
sys.path.insert(0, str(REPO_ROOT / "08_la2a"))            # dataset_la2a, splits_la2a
sys.path.insert(0, str(REPO_ROOT / "03_initial_GR_pred")) # eval_helpers

from amplitude_match import GR_DB_MIN, GR_DB_MAX, amplitude_match
from model_gainprior_ws import GainPriorWSDiffSSLLSTM
from system_gainprior import esr_metric
from splits_la2a import (
    La2aSplitManifest, LA2A_PARAM_ORDER, normalize_la2a_params, region_bounds,
)
from dataset_la2a import discover_la2a_pairs, RMS_WINDOW, SAMPLE_RATE
from eval_helpers import (
    _checkpoint_sort_key, _find_hparams_json, _pair_num_frames,
    _read_dry_wet_segment, _weighted_average_metric_rows, latest_best_checkpoint,
    list_runs, plot_loss_curves,
)
from src.dsp_torch import gain_reduction_db

DATA_ROOT = "/Volumes/Saola's Drive/AllCode/thesis/data/LA2A"
RUNS_DIR = "/Volumes/Saola's Drive/AllCode/thesis/data/la2a_gain_prior_runs"
DEVICE = "cpu"

print(f"torch {torch.__version__}  |  device: {DEVICE}")
print(f"GR clamp: [{GR_DB_MIN}, {GR_DB_MAX}] dB  |  RMS window: {RMS_WINDOW} samples")

In [ ]:
# ── 1. List available gain-prior runs ────────────────────────────────

MODEL_TYPE = "GainPriorWSDiffSSLLSTM"   # hparams["model_type"]

runs_df = list_runs(RUNS_DIR)
assert not runs_df.empty, f"No runs in {RUNS_DIR}"
available_models_df = runs_df[runs_df["type"] == MODEL_TYPE].copy()
if available_models_df.empty:
    # fall back to folder-name tag if hparams["model_type"] is missing/older
    available_models_df = runs_df[runs_df["run"].str.contains("gain_prior_ws", case=False, na=False)].copy()
assert not available_models_df.empty, (
    f"No '{MODEL_TYPE}' runs in {RUNS_DIR} — train with train_lstm_gain_prior_ws.ipynb first."
)

display_cols = ["run", "type", "epoch", "best_val", "n_ckpts", "eval_ckpt", "modified"]
print(f"Available gain-prior (waveshaper) runs: {len(available_models_df)}")
display(available_models_df[display_cols])

In [ ]:
# ── 2. Select run, checkpoint and split manifest ─────────────────────

# Leave as None to auto-pick the most recently modified matching run.
SELECTED_RUN_NAME: str | None = None

if not SELECTED_RUN_NAME:
    selected_run = available_models_df.iloc[-1]   # list_runs sorts by mtime asc
else:
    matches = available_models_df[available_models_df["run"] == SELECTED_RUN_NAME]
    assert len(matches) == 1, f"Run not found: {SELECTED_RUN_NAME}"
    selected_run = matches.iloc[0]

RUN_NAME = str(selected_run["run"])
RUN_DIR = os.path.join(RUNS_DIR, RUN_NAME)
CKPT_DIR = os.path.join(RUN_DIR, "checkpoints")
HPARAMS_PATH = os.path.join(RUN_DIR, "hparams.json")
EVAL_CKPT_PATH = latest_best_checkpoint(RUN_DIR)

with open(HPARAMS_PATH) as f:
    HP = json.load(f)
SPLIT = La2aSplitManifest.load(os.path.join(RUN_DIR, "split_manifest.json"))
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = SPLIT.train_frac, SPLIT.val_frac, SPLIT.test_frac

# LA2A split is temporal WITHIN each recording (every setting in every split), so
# "val"/"test" are the [.8,.9)/[.9,1) fraction REGIONS of all recordings, not
# held-out (song, setting) pairs. Discover recordings (paths+frames) and match to
# the manifest inventory by id.
RECS = discover_la2a_pairs(DATA_ROOT)
REC_BY_ID = {r["id"]: r for r in RECS}
_man_ids = {p["id"] for p in SPLIT.pairs}
_missing = _man_ids - set(REC_BY_ID)
assert not _missing, f"{len(_missing)} manifest recordings missing on disk: {sorted(_missing)}"
EVAL_RECS = [REC_BY_ID[p["id"]] for p in sorted(SPLIT.pairs, key=lambda q: q["id"])]


def _region(rec, split):
    return region_bounds(rec["frames"], TRAIN_FRAC, VAL_FRAC, TEST_FRAC)[split]


def _split_seconds(split):
    return sum(_region(r, split)[1] - _region(r, split)[0] for r in EVAL_RECS) / int(HP["sample_rate"])


print(f"Selected run   : {RUN_NAME}")
print(f"Eval checkpoint: {os.path.basename(EVAL_CKPT_PATH)}")
if pd.notna(selected_run.get("best_val")):
    print(f"Best val loss  : {selected_run['best_val']:.6f}")
print(f"Recordings     : {len(EVAL_RECS)}  ({len(SPLIT.settings)} unique [comp_limit, peak_reduction])")
print(f"Split policy   : temporal within recording — "
      f"train[0,{TRAIN_FRAC}) val[{TRAIN_FRAC},{round(TRAIN_FRAC+VAL_FRAC,3)}) "
      f"test[{round(TRAIN_FRAC+VAL_FRAC,3)},1)")
print(f"Val  regions   : {_split_seconds('val')/60:.1f} min total (unseen audio, known settings)")
print(f"Test regions   : {_split_seconds('test')/60:.1f} min total")

In [ ]:
# ── 3. Inspect checkpoints ──────────────────────────────────────────

print(f"Selected run   : {RUN_NAME}")
print(f"Eval checkpoint: {os.path.basename(EVAL_CKPT_PATH)}")
print("Checkpoints    :")
for c in sorted(glob.glob(os.path.join(CKPT_DIR, "*.ckpt")), key=_checkpoint_sort_key):
    marker = " <-- eval" if os.path.abspath(c) == os.path.abspath(EVAL_CKPT_PATH) else ""
    sz = os.path.getsize(c) / 1e6
    print(f"  {os.path.basename(c):<36}  {sz:6.2f} MB{marker}")

In [ ]:
# ── 4. Loss curves (total + components + gain-head activity) ─────────

_ = plot_loss_curves(RUN_DIR)

csv_path = Path(RUN_DIR) / "csv" / "metrics.csv"
if csv_path.exists():
    _df = pd.read_csv(csv_path)

    comp_cols = [
        ("loss/val_td", "L1 (val)"),
        ("loss/val_fd", "MR-STFT (val)"),
        ("loss/val_env", "env-dB (val)"),
        ("loss/val_pe", "pre-emph (val)"),
        ("esr/val", "ESR (val)"),
        ("mae/val", "MAE (val)"),
    ]
    if any(c in _df.columns for c, _ in comp_cols):
        fig, ax = plt.subplots(figsize=(8, 4))
        for col, label in comp_cols:
            if col in _df.columns:
                rows = _df.dropna(subset=[col])[["epoch", col]]
                if len(rows):
                    ax.plot(rows["epoch"], rows[col], label=label, lw=1.5)
        ax.set_xlabel("epoch"); ax.set_ylabel("metric"); ax.set_yscale("log")
        ax.grid(alpha=0.3, which="both"); ax.legend()
        ax.set_title(f"{RUN_NAME} — loss components (val)")
        plt.tight_layout(); plt.show()

    # gain-head activity: how far the model moves off the amplitude-matched
    # baseline (both start at exactly 0 by construction).
    gain_cols = [("gain/val_delta_db", "mean |Δg| (dB, val)"),
                 ("gain/val_color", "mean |color| (val)")]
    if any(c in _df.columns for c, _ in gain_cols):
        fig, ax = plt.subplots(figsize=(8, 3))
        ax2 = ax.twinx()
        for (col, label), a, color in zip(gain_cols, (ax, ax2), ("#d62728", "#1f77b4")):
            if col in _df.columns:
                rows = _df.dropna(subset=[col])[["epoch", col]]
                if len(rows):
                    a.plot(rows["epoch"], rows[col], label=label, lw=1.5, color=color)
                    a.set_ylabel(label, color=color)
        ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
        ax.set_title(f"{RUN_NAME} — gain-head activity (0 == amplitude-matched baseline)")
        plt.tight_layout(); plt.show()

In [ ]:
# ── 5. Checkpoint loading (GainPriorWSDiffSSLLSTM) ───────────────────

def load_model_from_ckpt(ckpt_path, hparams_path=None):
    # Build from hparams.json and load weights (strip Lightning 'model.' prefix).
    hparams_path = hparams_path or _find_hparams_json(ckpt_path)
    with open(hparams_path) as f:
        hp = json.load(f)
    assert hp.get("model_type") == MODEL_TYPE, hp.get("model_type")
    m = hp["model"]
    model = GainPriorWSDiffSSLLSTM(
        num_controls=int(m["num_controls"]),
        hidden_size=int(m["hidden_size"]),
        num_layers=int(m["num_layers"]),
        tvcond_dim=int(m.get("tvcond_dim", 16)),
        cond_block_size=int(m.get("cond_block_size", 128)),
        cond_num_layers=int(m.get("cond_num_layers", 1)),
        delta_max_db=float(m.get("delta_max_db", 12.0)),
        use_color=bool(m.get("use_color", True)),
        ws_hidden=int(m.get("ws_hidden", 8)),
        ws_film=bool(m.get("ws_film", False)),
    )
    sd = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)["state_dict"]
    model.load_state_dict({k[len("model."):]: v for k, v in sd.items() if k.startswith("model.")})
    model.to(DEVICE).eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(
        f"Loaded {n_params:,}-param GainPriorWSDiffSSLLSTM  "
        f"(hidden={model.hidden_size}, controls={model.num_controls}, "
        f"cond_block={model.cond_block_size}, delta_max={model.delta_max_db} dB, "
        f"color={model.use_color}, ws_film={model.ws_film})"
    )
    return model, hp


MODEL, HP = load_model_from_ckpt(EVAL_CKPT_PATH, HPARAMS_PATH)
SR = int(HP["sample_rate"])
SAMPLE_LENGTH = int(HP.get("sample_length", 132300))
# GR derived from waveforms uses the same 1024-sample causal RMS window the
# exported gr_curves were built with (== dataset_la2a on-the-fly GR).
RMS_WIN = int(HP.get("rms_window", RMS_WINDOW))

In [ ]:
# ── 5b. Learned transfer curve W(s) — the coloration mechanism ───────
# The waveshaper is where harmonics come from (composition, not additive
# synthesis): plot W(s) vs identity and the deviation W(s)−s. Bowing away from
# identity at high |s| = saturation; asymmetry = even harmonics. With ws_film
# on, the curve is programme-modulated — this is the UNMODULATED static curve.

_s = torch.linspace(-1.0, 1.0, 1001, device=DEVICE).view(1, 1, -1)
with torch.no_grad():
    _w = MODEL.waveshaper(_s).cpu().flatten().numpy()
_s = _s.cpu().flatten().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(_s, _s, "--", color="gray", lw=0.8, label="identity")
ax1.plot(_s, _w, color="#d62728", lw=1.2, label="W(s)")
ax1.set_xlabel("in"); ax1.set_ylabel("out"); ax1.set_title("Learned transfer curve")
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
ax2.plot(_s, _w - _s, color="#d62728", lw=1.2)
ax2.axhline(0, color="k", lw=0.5, alpha=0.4)
ax2.set_xlabel("in"); ax2.set_ylabel("W(s) − s")
ax2.set_title(f"Deviation from identity (max {np.abs(_w - _s).max():.4f})")
ax2.grid(alpha=0.3)
fig.suptitle(f"{RUN_NAME} — waveshaper", fontsize=10)
plt.tight_layout(); plt.show()


In [ ]:
# ── 6. Prediction & visualisation (stateful, gain-prior + waveshaper) ─
# LA2A data access: paths by recording id, static knobs [comp_limit, peak_reduction],
# and GR recomputed ON-THE-FLY from (dry, wet) with the exact export function
# (causal 1024-RMS + 1023-sample lookback == the exported .pt). A pre_roll_sec
# context warms the LSTM before the scored segment; return_parts_full exposes the
# model's own decomposition (Δg, waveshaper residual W(s)−s, additive c).

DEFAULT_PRE_ROLL_SEC = 10.0
LOOKBACK = RMS_WIN - 1


def _params_for(rec) -> torch.Tensor:
    # Normalised static knobs [1, 2] from one recording's setting.
    return torch.tensor(
        normalize_la2a_params(rec["comp_limit"], rec["peak_reduction"]),
        dtype=torch.float32, device=DEVICE,
    ).unsqueeze(0)


def _read_seg_with_gr(rec, start: int, stop: int):
    # dry, gr, wet for [start, stop) as [1, n] tensors. GR recomputed on-the-fly
    # with a 1023-sample lookback so the causal window is filled from real samples
    # (bit-identical to slicing the exported .pt).
    read_start = max(0, start - LOOKBACK)
    dry, wet = _read_dry_wet_segment(rec["dry"], rec["wet"], read_start, stop, SR)
    n = min(dry.shape[-1], wet.shape[-1])
    dry, wet = dry[..., :n], wet[..., :n]
    gr = gain_reduction_db(dry, wet, RMS_WIN)
    drop = start - read_start
    return dry[..., drop:], gr[..., drop:], wet[..., drop:]


def _gr_from_audio(dry: torch.Tensor, sig: torch.Tensor) -> np.ndarray:
    # GR (dB) of `sig` relative to `dry`, clamped to the GR range (GR-bins definition).
    return gain_reduction_db(dry, sig, RMS_WIN).clamp(GR_DB_MIN, GR_DB_MAX).squeeze().cpu().numpy()


def _default_test_segment_sec(rec) -> float:
    # A segment start inside the recording's TEST region (region start + 30 s, clamped).
    r0, r1 = _region(rec, "test")
    start = min(r0 + int(30 * SR), max(r0, r1 - int(6 * SR)))
    return start / SR


@torch.no_grad()
def predict_wet_segment(rec, start_sec, duration_sec=6.0,
                        pre_roll_sec=DEFAULT_PRE_ROLL_SEC, model=None):
    model = model or MODEL
    if isinstance(rec, int):
        rec = REC_BY_ID[rec]
    start = int(round(start_sec * SR))
    stop = start + int(round(duration_sec * SR))
    context_start = max(0, start - int(round(pre_roll_sec * SR)))
    offset = start - context_start

    dry_ctx, gr_ctx, wet_ctx = _read_seg_with_gr(rec, context_start, stop)
    params = _params_for(rec)

    model.reset_states()
    pred_full, delta_full, color_full, ws_full = model(
        dry_ctx.unsqueeze(0).to(DEVICE), gr_ctx.unsqueeze(0).to(DEVICE), params,
        return_parts_full=True)

    seg_len = min(stop - start, pred_full.shape[-1] - offset)
    sl = slice(offset, offset + seg_len)
    pred = pred_full[..., sl].squeeze(0).cpu()
    delta = delta_full[..., sl].squeeze(0).cpu()
    color = color_full[..., sl].squeeze(0).cpu()
    ws = ws_full[..., sl].squeeze(0).cpu()
    wet = wet_ctx[..., sl].cpu()
    dry = dry_ctx[..., sl].cpu()
    gr_in = gr_ctx[..., sl].cpu()
    err = (pred - wet).squeeze().numpy()

    gr_true = _gr_from_audio(dry, wet)
    gr_pred = _gr_from_audio(dry, pred)
    gr_err = gr_pred - gr_true

    t_len = pred.shape[-1]
    return {
        "id": rec["id"], "comp_limit": rec["comp_limit"], "peak_reduction": rec["peak_reduction"],
        "label": f"id {rec['id']} · cl={rec['comp_limit']} pr={rec['peak_reduction']}",
        "sample_rate": SR, "start_sec": start_sec, "duration_sec": t_len / SR,
        "pre_roll_sec": offset / SR, "time": np.arange(t_len) / SR,
        "dry": dry.squeeze().numpy(), "wet": wet.squeeze().numpy(), "pred": pred.squeeze().numpy(),
        "error": err, "gr_in": gr_in.squeeze().numpy(),
        "delta_db": delta.squeeze().numpy(), "color": color.squeeze().numpy(),
        "ws_res": ws.squeeze().numpy(),
        "gr_true": gr_true, "gr_pred": gr_pred, "gr_error": gr_err,
        "mae": float(np.mean(np.abs(err))),
        "esr": float(esr_metric(wet.unsqueeze(0), pred.unsqueeze(0))),
        "gr_mae_db": float(np.mean(np.abs(gr_err))),
        "gr_rmse_db": float(np.sqrt(np.mean(gr_err ** 2))),
        "delta_mae_db": float(np.mean(np.abs(delta.numpy()))),
        "color_mae": float(np.mean(np.abs(color.numpy()))),
        "ws_mae": float(np.mean(np.abs(ws.numpy()))),
    }


def visualize_prediction(rec=None, start_sec=None, duration_sec=6.0,
                         pre_roll_sec=DEFAULT_PRE_ROLL_SEC, title=None):
    rec = EVAL_RECS[0] if rec is None else (REC_BY_ID[rec] if isinstance(rec, int) else rec)
    start_sec = _default_test_segment_sec(rec) if start_sec is None else start_sec
    seg = predict_wet_segment(rec, start_sec, duration_sec, pre_roll_sec)

    t = seg["time"]
    # rows: dry | waveform (wet/pred) | waveform error + waveshaper residual
    #       | GR (true/pred) | GR err | gain decomposition (GR input vs GR+Δg)
    fig, axes = plt.subplots(6, 1, figsize=(11, 15.5), sharex=True,
                             gridspec_kw={"height_ratios": [1, 2, 1, 2, 1, 2]})

    axes[0].plot(t, seg["dry"], lw=0.6, color="#444")
    axes[0].set_ylabel("dry"); axes[0].set_ylim(-1.05, 1.05); axes[0].grid(alpha=0.3)

    axes[1].plot(t, seg["wet"], label="target (wet)", lw=1.0, color="#1f77b4")
    axes[1].plot(t, seg["pred"], label="prediction", lw=0.9, color="#d62728", alpha=0.85)
    axes[1].set_ylabel("amp"); axes[1].legend(loc="lower right"); axes[1].set_ylim(-1.05, 1.05)
    axes[1].grid(alpha=0.3)

    axes[2].plot(t, seg["error"], lw=0.8, color="#2ca02c", label="wave err")
    axes[2].plot(t, seg["ws_res"], lw=0.7, color="#d62728", alpha=0.6, label="W(s)−s")
    axes[2].axhline(0, color="k", lw=0.5, alpha=0.4)
    axes[2].set_ylabel("wave err"); axes[2].legend(loc="lower right", fontsize=8)
    axes[2].grid(alpha=0.3)

    axes[3].plot(t, seg["gr_true"], label="target", lw=1.4, color="#1f77b4")
    axes[3].plot(t, seg["gr_pred"], label="prediction", lw=1.2, color="#d62728", alpha=0.85)
    axes[3].axhline(0, color="k", lw=0.5, alpha=0.4)
    axes[3].set_ylabel("GR (dB)"); axes[3].legend(loc="lower right"); axes[3].grid(alpha=0.3)

    axes[4].plot(t, seg["gr_error"], lw=0.8, color="#9467bd")
    axes[4].axhline(0, color="k", lw=0.5, alpha=0.4)
    axes[4].set_ylabel("GR err (dB)"); axes[4].grid(alpha=0.3)

    axes[5].plot(t, seg["gr_in"], label="GR input (prior)", lw=1.0, color="#1f77b4")
    axes[5].plot(t, seg["gr_in"] + seg["delta_db"], label="GR + learned Δg", lw=0.9,
                 color="#d62728", alpha=0.85)
    axes[5].plot(t, seg["delta_db"], label="Δg alone (dB)", lw=0.7, color="#ff7f0e", alpha=0.7)
    axes[5].axhline(0, color="k", lw=0.5, alpha=0.4)
    axes[5].set_ylabel("gain (dB)"); axes[5].set_xlabel("time (s)")
    axes[5].legend(loc="lower right", fontsize=8); axes[5].grid(alpha=0.3)

    fig.suptitle(title or (
        f"{seg['label']}   (TEST region, {seg['start_sec']:.0f}s)\n"
        f"pre-roll={seg['pre_roll_sec']:.1f}s  MAE {seg['mae']:.4f}  ESR {seg['esr']:.4f}  "
        f"GR MAE={seg['gr_mae_db']:.2f} dB  GR RMSE={seg['gr_rmse_db']:.2f} dB  "
        f"mean|Δg|={seg['delta_mae_db']:.3f} dB  mean|ws|={seg['ws_mae']:.5f}  "
        f"mean|c|={seg['color_mae']:.5f}"
    ), fontsize=10)
    plt.tight_layout(); plt.show()
    return seg

## Usage

Re-run the cells below any time; they read the latest synced run outputs from Drive.
Use the available-runs cell first, then set `SELECTED_RUN_NAME` in the selection cell.

- `visualize_prediction(rec=...)` — any recording (an id, or a rec dict); defaults to
  the first recording and a 6 s segment inside its **test** region. Shows the waveform
  (target / prediction), the gain-reduction curve recovered from the prediction, **and
  the model's own decomposition** — the GR input vs `GR+Δg` (where the model overrides
  the prior; concentrates at attack transients) and the coloration share.
- `pr_sweep(comp_limit=0)` — test-region segments across a spread of Peak-Reduction
  values (LA2A analog of the diffssl settings sweep; each pr is a **different**
  recording, so the audio differs). Table adds `mean |Δg|`.
- The split-metrics cells stream every recording's **val** / **test** region on the
  canonical 9-column engine (ported from `06_output/eval_lstm_gain_prior_ws.ipynb`) —
  results duration-weighted and CSV-saved next to the checkpoint.
- §9 runs the GR-source ablation (oracle / const-mean / const-0 dB / amp-match).

In [ ]:
# Single prediction preview — a 6 s segment inside the TEST region of one recording
_ = visualize_prediction(rec=EVAL_RECS[0], duration_sec=6.0)

In [ ]:
# ── Peak-reduction sweep (LA2A analog of settings_sweep) ─────────────
# Each LA2A setting is a DIFFERENT recording (different audio), so this shows the
# model tracking increasing compression across recordings — not one song
# reprocessed. Picks comp_limit=SWEEP_CL and a spread of Peak-Reduction values,
# each on a 6 s segment inside its own test region.

SWEEP_CL = 0


def pr_sweep(comp_limit=SWEEP_CL, pr_values=(0, 25, 50, 75, 100),
             duration_sec=6.0, pre_roll_sec=DEFAULT_PRE_ROLL_SEC):
    recs = []
    for pr in pr_values:
        cand = [r for r in EVAL_RECS if r["comp_limit"] == comp_limit and r["peak_reduction"] == pr]
        if cand:
            recs.append(cand[0])
    segs = [predict_wet_segment(r, _default_test_segment_sec(r), duration_sec, pre_roll_sec)
            for r in recs]

    fig, axes = plt.subplots(len(segs), 1, figsize=(11, 1.8 * len(segs)),
                             sharex=True, squeeze=False)
    for ax, s in zip(axes[:, 0], segs):
        ax.plot(s["time"], s["gr_true"], lw=1.4, color="#1f77b4", label="target")
        ax.plot(s["time"], s["gr_pred"], lw=1.2, color="#d62728", alpha=0.85, label="prediction")
        ax.plot(s["time"], s["gr_in"], lw=0.9, ls="--", color="#7f7f7f", alpha=0.8, label="GR input")
        ax.axhline(0, color="k", lw=0.5, alpha=0.4)
        ax.set_ylabel("GR (dB)", fontsize=8)
        ax.set_title(
            f"{s['label']}  —  GR MAE {s['gr_mae_db']:.2f} dB  "
            f"MAE {s['mae']:.4f}  ESR {s['esr']:.4f}  "
            f"mean|Δg| {s['delta_mae_db']:.3f} dB  mean|ws| {s['ws_mae']:.5f}",
            fontsize=8, loc="left")
        ax.grid(alpha=0.3)
    axes[0, 0].legend(loc="lower right", fontsize=8)
    axes[-1, 0].set_xlabel("time (s)")
    fig.suptitle(f"Peak-reduction sweep — GR curves — comp_limit={comp_limit} "
                 f"({duration_sec:.0f}s test-region segments)", fontsize=11)
    plt.tight_layout(); plt.show()

    df = pd.DataFrame([
        {"comp_limit": s["comp_limit"], "peak_reduction": s["peak_reduction"],
         "GR MAE (dB)": s["gr_mae_db"], "MAE": s["mae"], "ESR": s["esr"],
         "mean |dg| (dB)": s["delta_mae_db"], "mean |ws|": s["ws_mae"], "mean |color|": s["color_mae"]}
        for s in segs
    ])
    display(df)
    return segs, df


sweep_segs, sweep_df = pr_sweep()

In [ ]:
# ── 6b. Metric engine (ported from 02b_sota_training/eval_lstm_diffssl_tvc.ipynb) ──
#
# The canonical 9-column table so this model drops straight into the cross-model
# comparison. Every moving-average envelope uses an O(N) cumsum window; the FFT
# metrics reimplement auraloss MultiResolutionSTFTLoss and src.losses.{M_SF, EDC},
# batched. MR-STFT and ESR are verified below against nablafx.evaluation (auraloss)
# to <1e-3 / <1e-4.

from scipy.signal import bilinear, lfilter

EPS = 1e-10

# Multi-resolution window / FFT sets (identical to the diffssl_tvc engine)
MR_STE_WINDOWS   = (256, 1024, 4096)                     # short-time energy (envelope shape)
MR_NRMSE_WINDOWS = (512, 1024, 2048)                     # == src.losses.multi_resolution_nrmse
STFT_SIZES       = (512, 1024, 2048)                     # spectral-flux resolutions
_MRSTFT_RES      = [(1024, 120, 600), (2048, 240, 1200), (512, 50, 240)]   # auraloss defaults

# Final metric columns: GR-stage | colour-stage | standard
METRIC_COLS = [
    "GR MAE (dB)", "MR-STE",                             # GR stage
    "MR-STFT", "ESR (A-wt)",                             # colour stage (auraloss MR-STFT + A-weighted ESR)
    "MAE (L1)", "MSE (L2)", "EDC", "M_NRMSE", "M_SF",    # standard
]


# ── moving-average envelopes (cumsum: O(N), window-independent) ──
def _moving_meansq(x_sq, W, centered):
    """Sliding mean of x**2 via prefix sums. centered=symmetric, else causal (trailing)."""
    n = x_sq.shape[-1]
    c = np.empty(n + 1); c[0] = 0.0; np.cumsum(x_sq, out=c[1:])
    i = np.arange(n)
    if centered:
        lo = i - (W // 2); hi = lo + W
    else:                                                # causal == src.dsp_torch.gain_reduction_db
        hi = i + 1; lo = hi - W
    lo = np.clip(lo, 0, n); hi = np.clip(hi, 0, n)
    return (c[hi] - c[lo]) / np.maximum(hi - lo, 1)

def _to_db(x):       return 20.0 * np.log10(np.maximum(x, EPS))
def _esr(pred, tgt): return float(np.sum((tgt - pred) ** 2) / (np.sum(tgt * tgt) + 1e-8))

# A-weighting IIR (analog prototype -> bilinear) for the A-weighted ESR
def _a_weighting_ba(fs):
    f1, f2, f3, f4 = 20.598997, 107.65265, 737.86223, 12194.217
    A1000 = 1.9997
    nums = [(2 * np.pi * f4) ** 2 * 10 ** (A1000 / 20.0), 0, 0, 0, 0]
    dens = np.polymul([1, 4 * np.pi * f4, (2 * np.pi * f4) ** 2],
                      [1, 4 * np.pi * f1, (2 * np.pi * f1) ** 2])
    dens = np.polymul(np.polymul(dens, [1, 2 * np.pi * f3]), [1, 2 * np.pi * f2])
    return bilinear(nums, dens, fs)
AW_B, AW_A = _a_weighting_ba(SR)


def reference_gr_db(dry, wet, W=RMS_WIN):
    """Reference GR trajectory = causal RMS GR (matches src.dsp_torch.gain_reduction_db)."""
    return (_to_db(np.sqrt(_moving_meansq(wet * wet, W, centered=False)))
            - _to_db(np.sqrt(_moving_meansq(dry * dry, W, centered=False))))


# ── GR-stage / envelope metrics ──
def mr_ste(pred, wet, windows=MR_STE_WINDOWS):
    """Multi-resolution short-time energy distance (envelope shape; Wright & Valimaki)."""
    pe, we = pred * pred, wet * wet
    total = 0.0
    for W in windows:
        ep = _moving_meansq(pe, W, centered=True)
        et = _moving_meansq(we, W, centered=True)
        total += np.sum(np.abs(ep - et)) / (np.sum(np.abs(et)) + 1e-8)
    return float(total / len(windows))

def mr_nrmse(pred, wet, windows=MR_NRMSE_WINDOWS):
    """Mean normalised RMS-envelope error (== src.losses.multi_resolution_nrmse, cumsum-fast)."""
    total = 0.0
    for W in windows:
        rp = np.sqrt(_moving_meansq(pred * pred, W, centered=False))
        rt = np.sqrt(_moving_meansq(wet * wet, W, centered=False))
        total += np.sqrt(np.mean((rt - rp) ** 2)) / (np.sqrt(np.mean(rt * rt)) + 1e-8)
    return float(total / len(windows))


# ── colour-stage + standard FFT metrics ──
_HANN = {}
def _win(n):
    if n not in _HANN:
        _HANN[n] = torch.hann_window(n)
    return _HANN[n]

def _stft_mag_pow(x2d, n_fft, hop, win):
    """auraloss-style magnitude = sqrt(clamp(re^2 + im^2, 1e-8))."""
    st = torch.stft(x2d, n_fft=n_fft, hop_length=hop, win_length=win,
                    window=_win(win), return_complex=True)
    return torch.sqrt(torch.clamp(st.real ** 2 + st.imag ** 2, min=1e-8))

def _stft_mag_fft(x2d, n_fft):
    """src.losses-style magnitude (hop = n_fft // 4, hann(n_fft))."""
    return torch.stft(x2d, n_fft=n_fft, hop_length=n_fft // 4,
                      window=_win(n_fft), return_complex=True).abs()

@torch.no_grad()
def fft_metrics(pred1d: torch.Tensor, wet1d: torch.Tensor) -> dict:
    """auraloss MR-STFT + src.losses {M_SF, EDC} for one (pred, wet) pair."""
    P1, T1 = pred1d.unsqueeze(0), wet1d.unsqueeze(0)
    out = {}

    # MR-STFT (auraloss default = spectral-convergence + log-magnitude, mean over 3 resolutions)
    acc = 0.0
    for n_fft, hop, win in _MRSTFT_RES:
        P = _stft_mag_pow(P1, n_fft, hop, win)[0]
        T = _stft_mag_pow(T1, n_fft, hop, win)[0]
        sc = torch.linalg.norm(T - P) / torch.linalg.norm(T)
        lm = (torch.log(P) - torch.log(T)).abs().mean()
        acc += float(sc + lm)
    out["MR-STFT"] = acc / len(_MRSTFT_RES)

    # spectral-flux (M_SF) over (512, 1024, 2048)
    sf = 0.0
    for n_fft in STFT_SIZES:
        P = _stft_mag_fft(P1, n_fft)[0]
        T = _stft_mag_fft(T1, n_fft)[0]
        pf, tf = torch.diff(P, dim=-1), torch.diff(T, dim=-1)
        sf += float((tf - pf).abs().mean() / (tf.abs().mean() + 1e-8))
    out["M_SF"] = sf / len(STFT_SIZES)

    # EDC (energy-decay-curve error, dB)
    pe = torch.flip(torch.cumsum(torch.flip(P1 ** 2, dims=(-1,)), dim=-1), dims=(-1,))
    te = torch.flip(torch.cumsum(torch.flip(T1 ** 2, dims=(-1,)), dim=-1), dims=(-1,))
    pe = pe / (pe[..., :1] + 1e-8); te = te / (te[..., :1] + 1e-8)
    out["EDC"] = float((10 * torch.log10(te.clamp(min=1e-8))
                        - 10 * torch.log10(pe.clamp(min=1e-8))).abs().mean())
    return out


def chunk_metrics(dry: np.ndarray, pred: np.ndarray, wet: np.ndarray) -> dict:
    """The 9 metric columns for one (dry, pred, wet) chunk (float64 1-D arrays).

    GR MAE compares the prediction's causal-RMS GR against the reference RMS
    GR(dry, wet), both clamped to [GR_DB_MIN, GR_DB_MAX]. FFT metrics run in float32.
    """
    gr_tgt  = np.clip(reference_gr_db(dry, wet),  GR_DB_MIN, GR_DB_MAX)
    gr_pred = np.clip(reference_gr_db(dry, pred), GR_DB_MIN, GR_DB_MAX)
    row = {
        "GR MAE (dB)": float(np.mean(np.abs(gr_pred - gr_tgt))),
        "MR-STE":      mr_ste(pred, wet),
        "ESR (A-wt)":  _esr(lfilter(AW_B, AW_A, pred), lfilter(AW_B, AW_A, wet)),
        "MAE (L1)":    float(np.mean(np.abs(pred - wet))),
        "MSE (L2)":    float(np.mean((pred - wet) ** 2)),
        "M_NRMSE":     mr_nrmse(pred, wet),
    }
    row.update(fft_metrics(torch.from_numpy(pred).float(),
                           torch.from_numpy(wet).float()))
    return row


# ── verify MR-STFT / ESR against nablafx.evaluation (auraloss) ──
try:
    from nablafx.evaluation import get_function
    _g = torch.Generator().manual_seed(0)
    _p = torch.randn(SR, generator=_g); _t = torch.randn(SR, generator=_g)
    _nab_mrstft, _nab_esr = get_function("mrstft_loss"), get_function("esr_loss")
    _pt, _tt = _p.view(1, 1, -1), _t.view(1, 1, -1)
    _fm = fft_metrics(_p, _t)
    _d_mrstft = abs(float(_nab_mrstft(_pt, _tt)) - _fm["MR-STFT"])
    _d_esr = abs(float(_nab_esr(_pt, _tt)) - _esr(_p.double().numpy(), _t.double().numpy()))
    assert _d_mrstft < 1e-3, _d_mrstft
    assert _d_esr < 1e-4, _d_esr
    print(f"nablafx.evaluation verified — MR-STFT Δ={_d_mrstft:.2e}, ESR Δ={_d_esr:.2e}")
except Exception as e:
    print(f"nablafx verification skipped: {e}")

print(f"Metric engine registered — {len(METRIC_COLS)} columns: {METRIC_COLS}")

In [ ]:
# ── 7. Region streaming engine + VALIDATION split metrics ────────────
# For every recording, stream its VAL region [.8,.9) in non-overlapping chunks with
# LSTM state carried across chunks (main LSTM + tvcond). reset_states() once at the
# region start; PRE_ROLL_CHUNKS warm-up chunk(s) are fed but not scored (warms the
# state; also fills the GR lookback). GR recomputed on-the-fly per region buffer.
# Each scored chunk goes through the ported 9-column engine, duration-weighted.

MAX_EVAL_RECS: int | None = None       # small int for a smoke test
REGION_EVAL_SEC: float | None = 30.0   # scored seconds per region (None = whole region)
STREAM_CHUNK_SEC = 10.0
PRE_ROLL_CHUNKS = 1                     # warm-up chunks fed before scoring
SAVE_CHUNK_METRICS = False
MIN_METRIC_FRAMES = max(STFT_SIZES)     # skip sub-FFT tail chunks

sr = SR
_BLOCK = int(HP["model"].get("cond_block_size", 128))
chunk_samples = max(_BLOCK, (int(round(STREAM_CHUNK_SEC * sr)) // _BLOCK) * _BLOCK)


def _region_buffer(rec, split):
    # dry/gr/wet for [ctx, score_end) where ctx = region_start − warm-up. `warm`
    # is how many leading samples are warm-up (not scored).
    r0, r1 = _region(rec, split)
    score_end = r1 if REGION_EVAL_SEC is None else min(r1, r0 + int(round(REGION_EVAL_SEC * sr)))
    ctx = max(0, r0 - PRE_ROLL_CHUNKS * chunk_samples)
    dry_b, gr_b, wet_b = _read_seg_with_gr(rec, ctx, score_end)
    return dry_b, gr_b, wet_b, (r0 - ctx)


@torch.no_grad()
def _stream_buffer(dry_b, gr_b, wet_b, warm, params, bypass=False):
    # Stateful chunked pass over one region buffer; per-chunk metric rows for the
    # scored part. bypass=True skips the network: pred = dry × 10^(gr/20).
    MODEL.reset_states()
    rows, N = [], dry_b.shape[-1]
    for o in range(0, N, chunk_samples):
        stop = min(o + chunk_samples, N)
        d, g, w = dry_b[..., o:stop], gr_b[..., o:stop], wet_b[..., o:stop]
        if bypass:
            pred = amplitude_match(d.unsqueeze(0), g.unsqueeze(0))
        else:
            pred = MODEL(d.unsqueeze(0).to(DEVICE), g.unsqueeze(0).to(DEVICE), params)
            MODEL.detach_states()
        if o < warm:                       # warm-up region — feed but don't score
            continue
        n = stop - o
        if n < MIN_METRIC_FRAMES:
            continue
        rows.append({"Start (s)": o / sr, "Duration (s)": n / sr, "Frames": n,
                     **chunk_metrics(d.squeeze(0).double().cpu().numpy(),
                                     pred.squeeze().double().cpu().numpy(),
                                     w.squeeze(0).double().cpu().numpy())})
    return rows


def stream_split_metrics(split, recs):
    chunk_rows, rec_rows = [], []
    for i, rec in enumerate(recs, start=1):
        print(f"[{split} {i}/{len(recs)}] id {rec['id']}  cl={rec['comp_limit']} pr={rec['peak_reduction']}")
        buf = _region_buffer(rec, split)
        rows = _stream_buffer(*buf, _params_for(rec))
        chunk_rows += [{"Id": rec["id"], "comp_limit": rec["comp_limit"],
                        "peak_reduction": rec["peak_reduction"], **r} for r in rows]
        rec_rows.append({"Split": split, "Id": rec["id"], "comp_limit": rec["comp_limit"],
                         "peak_reduction": rec["peak_reduction"],
                         "Duration (s)": sum(r["Frames"] for r in rows) / sr,
                         **_weighted_average_metric_rows(rows, METRIC_COLS)})
        gc.collect()
    total = sum(r["Frames"] for r in chunk_rows)
    split_row = {"Split": split, "Recordings": len(recs), "Chunks": len(chunk_rows),
                 "Duration (s)": total / sr, **_weighted_average_metric_rows(chunk_rows, METRIC_COLS)}
    return pd.DataFrame(rec_rows), pd.DataFrame([split_row]), chunk_rows


_val_recs = EVAL_RECS if MAX_EVAL_RECS is None else EVAL_RECS[:MAX_EVAL_RECS]
val_pair_metrics_df, val_audio_metrics_df, val_chunk_rows = stream_split_metrics("val", _val_recs)

val_pair_metrics_df.to_csv(Path(RUN_DIR) / "eval_audio_metrics_validation_pairs.csv", index=False)
val_audio_metrics_df.to_csv(Path(RUN_DIR) / "eval_audio_metrics_validation.csv", index=False)
if SAVE_CHUNK_METRICS:
    pd.DataFrame(val_chunk_rows).to_csv(Path(RUN_DIR) / "eval_audio_metrics_validation_chunks.csv", index=False)
print(f"Saved -> {RUN_DIR}/eval_audio_metrics_validation(.csv, _pairs.csv)")

display(val_pair_metrics_df)
display(val_audio_metrics_df)

## Held-out TEST split (final region of every recording)

The **test** split is the last `[.9, 1.0)` fraction of **every** recording — audio
the model never saw, at the same 42 settings it was trained on (LA2A's temporal-split
convention). Same stateful region streaming and same 9-column engine as the
validation cell; only the region differs. This is the headline generalisation number
(unseen audio, known settings). There is no separate external `test_ground_truth/`
set for LA2A — the SignalTrain corpus is a flat pool of recordings, not a
song × setting grid.

In [ ]:
# ── 8. Audio metrics on the TEST split ───────────────────────────────

_test_recs = EVAL_RECS if MAX_EVAL_RECS is None else EVAL_RECS[:MAX_EVAL_RECS]
test_pair_metrics_df, test_audio_metrics_df, test_chunk_rows = stream_split_metrics("test", _test_recs)

test_pair_metrics_df.to_csv(Path(RUN_DIR) / "eval_audio_metrics_test_pairs.csv", index=False)
test_audio_metrics_df.to_csv(Path(RUN_DIR) / "eval_audio_metrics_test.csv", index=False)
if SAVE_CHUNK_METRICS:
    pd.DataFrame(test_chunk_rows).to_csv(Path(RUN_DIR) / "eval_audio_metrics_test_chunks.csv", index=False)
print(f"Saved -> {RUN_DIR}/eval_audio_metrics_test(.csv, _pairs.csv)")

display(test_pair_metrics_df)
display(test_audio_metrics_df)

## 9. Conditioning ablations — what does the GR input actually buy? (no training)

Everything above conditions the model on the **oracle** GR (computed from the wet
target — unavailable at deployment). These no-training ablations stream the SAME
checkpoint over the SAME test regions with different GR sources:

| Variant | GR source | What it measures |
|---|---|---|
| `oracle` | on-the-fly GR from wet | upper bound (the numbers above) |
| `const_mean` | per-region mean of the oracle GR | static gain with the right average — isolates the value of the GR's *temporal* information |
| `const_0db` | 0 dB everywhere | no GR information; `y = W(x) + c` — the structural prior fully disabled |
| `amp_match` | oracle GR, **model bypassed** | `dry × 10^(gr/20)` scored with the identical engine — what the zero-init model does before training; the trained model must beat this to justify its heads |

No `predicted` variant: there is no LA2A GR predictor (05_conditioning's predictor is
diffssl-only). Reading the table: `oracle → const_mean` is the **temporal-GR value**;
`oracle → amp_match` is what the **network** (Δg + waveshaper + colour) adds on top of
the amplitude match; `const_0db` shows how much the model leans on the GR prior.

In [ ]:
# ── 9a. Run the GR-source ablation (few test regions, same ckpt) ─────
# Streams the SAME GainPriorWSDiffSSLLSTM checkpoint with different GR sources
# through the SAME 9-column engine, plus the model-bypassed amp_match baseline.
# No training anywhere.

ABLATION_VARIANTS = ["oracle", "const_mean", "const_0db", "amp_match"]
# a spread of settings (first available at comp_limit=0, pr in 0/50/100)
_abl_targets = [(0, 0), (0, 50), (0, 100)]
ABLATION_RECS = []
for cl, pr in _abl_targets:
    cand = [r for r in EVAL_RECS if r["comp_limit"] == cl and r["peak_reduction"] == pr]
    if cand:
        ABLATION_RECS.append(cand[0])
ABLATION_RECS = ABLATION_RECS or EVAL_RECS[:3]


def _variant_gr(gr_oracle, variant):
    if variant in ("oracle", "amp_match"):
        return gr_oracle
    if variant == "const_mean":
        return torch.full_like(gr_oracle, float(gr_oracle.mean()))
    if variant == "const_0db":
        return torch.zeros_like(gr_oracle)
    raise ValueError(variant)


abl_chunk_rows = {v: [] for v in ABLATION_VARIANTS}
abl_pair_rows = []
for i, rec in enumerate(ABLATION_RECS, start=1):
    dry_b, gr_or, wet_b, warm = _region_buffer(rec, "test")
    params = _params_for(rec)
    for variant in ABLATION_VARIANTS:
        print(f"[{i}/{len(ABLATION_RECS)}] id {rec['id']} pr={rec['peak_reduction']} — {variant}")
        gr_used = _variant_gr(gr_or, variant)
        rows = _stream_buffer(dry_b, gr_used, wet_b, warm, params, bypass=(variant == "amp_match"))
        abl_chunk_rows[variant] += rows
        abl_pair_rows.append({
            "Variant": variant, "Id": rec["id"], "peak_reduction": rec["peak_reduction"],
            "Frames": sum(r["Frames"] for r in rows),
            **_weighted_average_metric_rows(rows, METRIC_COLS)})
    gc.collect()

abl_split_df = pd.DataFrame([
    {"Variant": v, "Recordings": len(ABLATION_RECS),
     "Duration (s)": sum(r["Frames"] for r in abl_chunk_rows[v]) / sr,
     **_weighted_average_metric_rows(abl_chunk_rows[v], METRIC_COLS)}
    for v in ABLATION_VARIANTS
])
abl_pair_df = pd.DataFrame(abl_pair_rows)

abl_pair_df.to_csv(Path(RUN_DIR) / "eval_audio_metrics_gr_ablation_pairs.csv", index=False)
abl_split_df.to_csv(Path(RUN_DIR) / "eval_audio_metrics_gr_ablation.csv", index=False)
print(f"Saved -> {RUN_DIR}/eval_audio_metrics_gr_ablation(.csv, _pairs.csv)")

display(abl_split_df)
display(abl_pair_df)

In [ ]:
# ── 9b. Ablation GR curves — per-variant GR trajectory (one segment) ─
# For each ablation recording, run the same 6 s @ test-region segment (10 s pre-roll
# after reset_states) once per GR source and plot the GR curves: target (from wet)
# vs prediction (recovered from the variant output) vs the GR actually fed (dashed).

@torch.no_grad()
def predict_segment_variant(rec, start_sec, variant, duration_sec=6.0,
                            pre_roll_sec=DEFAULT_PRE_ROLL_SEC):
    start = int(round(start_sec * SR)); stop = start + int(round(duration_sec * SR))
    context_start = max(0, start - int(round(pre_roll_sec * SR)))
    offset = start - context_start
    dry_ctx, gr_or_ctx, wet_ctx = _read_seg_with_gr(rec, context_start, stop)
    gr_ctx = _variant_gr(gr_or_ctx, variant)
    bypass = variant == "amp_match"

    if bypass:
        pred_full = amplitude_match(dry_ctx.unsqueeze(0), gr_ctx.unsqueeze(0))
    else:
        MODEL.reset_states()
        pred_full = MODEL(dry_ctx.unsqueeze(0).to(DEVICE), gr_ctx.unsqueeze(0).to(DEVICE),
                          _params_for(rec))
    seg_len = min(stop - start, pred_full.shape[-1] - offset)
    sl = slice(offset, offset + seg_len)
    pred = pred_full[..., sl].squeeze(0).cpu()
    wet, dry = wet_ctx[..., sl].cpu(), dry_ctx[..., sl].cpu()
    gr_in = np.clip(gr_ctx[..., sl].squeeze().numpy(), GR_DB_MIN, GR_DB_MAX)
    err = (pred - wet).squeeze().numpy()
    gr_true, gr_pred = _gr_from_audio(dry, wet), _gr_from_audio(dry, pred)
    return {"variant": variant, "time": np.arange(pred.shape[-1]) / SR,
            "gr_in": gr_in, "gr_true": gr_true, "gr_pred": gr_pred,
            "mae": float(np.mean(np.abs(err))),
            "esr": float(esr_metric(wet.unsqueeze(0), pred.unsqueeze(0))),
            "gr_mae_db": float(np.mean(np.abs(gr_pred - gr_true))),
            "gr_in_mae_db": float(np.mean(np.abs(gr_in - gr_true)))}


def ablation_gr_sweep(rec, start_sec=None, duration_sec=6.0, pre_roll_sec=DEFAULT_PRE_ROLL_SEC):
    start_sec = _default_test_segment_sec(rec) if start_sec is None else start_sec
    segs = [predict_segment_variant(rec, start_sec, v, duration_sec, pre_roll_sec)
            for v in ABLATION_VARIANTS]

    fig, axes = plt.subplots(len(segs), 1, figsize=(11, 1.8 * len(segs)),
                             sharex=True, sharey=True, squeeze=False)
    for ax, s in zip(axes[:, 0], segs):
        ax.plot(s["time"], s["gr_true"], lw=1.4, color="#1f77b4", label="target")
        ax.plot(s["time"], s["gr_pred"], lw=1.2, color="#d62728", alpha=0.85, label="prediction")
        ax.plot(s["time"], s["gr_in"], lw=0.9, ls="--", color="#7f7f7f", alpha=0.8, label="GR input")
        ax.axhline(0, color="k", lw=0.5, alpha=0.4)
        ax.set_ylabel("GR (dB)", fontsize=8)
        ax.set_title(f"{s['variant']}  —  GR MAE {s['gr_mae_db']:.2f} dB  "
                     f"GR-input MAE {s['gr_in_mae_db']:.2f} dB  "
                     f"MAE {s['mae']:.4f}  ESR {s['esr']:.4f}", fontsize=8, loc="left")
        ax.grid(alpha=0.3)
    axes[0, 0].legend(loc="lower right", fontsize=8)
    axes[-1, 0].set_xlabel("time (s)")
    fig.suptitle(f"GR-source ablation — GR curves — id {rec['id']} "
                 f"cl={rec['comp_limit']} pr={rec['peak_reduction']} "
                 f"({duration_sec:.0f}s @ {start_sec:.0f}s)", fontsize=11)
    plt.tight_layout(); plt.show()

    df = pd.DataFrame([
        {"variant": s["variant"], "GR MAE (dB)": s["gr_mae_db"],
         "GR-input MAE (dB)": s["gr_in_mae_db"], "MAE": s["mae"], "ESR": s["esr"]}
        for s in segs])
    display(df)
    return segs, df


abl_sweeps = {rec["id"]: ablation_gr_sweep(rec) for rec in ABLATION_RECS}